# 21 — Scale Frustration: Predicting Pathology from K_nm Topology

When 3 mutually coupled scales have incommensurate frequencies, they
can't all synchronise pairwise — this creates FRUSTRATED TRIADS.

Prediction:
- Epilepsy = too FEW frustrated triads (everything locks → hypersync)
- Schizophrenia = too MANY frustrated triads (nothing locks → fragmentation)
- Healthy = intermediate frustration (flexible but coherent)

**Test**: Compute frustration index for the 16-scale K_nm network.
Then perturb K_nm to model specific pathologies and track how
frustration changes.

In [ ]:
import json

import numpy as np

from scpn_quantum_control.bridge import OMEGA_N_16, build_knm_paper27

In [ ]:
def compute_frustration(K, omega, threshold=0.01):
    """Compute frustration index of a coupled oscillator network.

    A triad (i,j,k) is frustrated if all three pairwise couplings exist
    but the frequency relationships make simultaneous sync impossible.

    Frustration condition: for triad (i,j,k), the phase differences
    at steady state satisfy theta_ij + theta_jk + theta_ki ≠ 0 mod 2pi.
    We approximate this by checking if omega differences are incommensurate.
    """
    N = len(omega)
    n_triads = 0
    n_frustrated = 0
    frustrated_triads = []

    for i in range(N):
        for j in range(i + 1, N):
            for k in range(j + 1, N):
                # All three pairs must be coupled
                if K[i, j] > threshold and K[j, k] > threshold and K[i, k] > threshold:
                    n_triads += 1

                    # Frustration: check if the three pairwise phase locks
                    # are mutually consistent
                    # At steady state: sin(theta_j - theta_i) = (omega_i - omega_j) / K_ij
                    # Triad is frustrated if |sum of steady-state phase diffs| > pi/2

                    d_ij = np.arcsin(np.clip((omega[i] - omega[j]) / (K[i, j] * N + 1e-10), -1, 1))
                    d_jk = np.arcsin(np.clip((omega[j] - omega[k]) / (K[j, k] * N + 1e-10), -1, 1))
                    d_ki = np.arcsin(np.clip((omega[k] - omega[i]) / (K[i, k] * N + 1e-10), -1, 1))

                    # Closure error: should be 0 for consistent triad
                    closure = abs(d_ij + d_jk + d_ki)

                    if closure > np.pi / 4:  # significantly inconsistent
                        n_frustrated += 1
                        frustrated_triads.append((i, j, k, closure))

    frustration_ratio = n_frustrated / max(n_triads, 1)
    return {
        "n_triads": n_triads,
        "n_frustrated": n_frustrated,
        "frustration_ratio": frustration_ratio,
        "top_frustrated": sorted(frustrated_triads, key=lambda x: -x[3])[:5],
    }


N = 16
K16 = build_knm_paper27(L=N)
omega16 = OMEGA_N_16[:N]

# Baseline: healthy system
healthy = compute_frustration(K16, omega16)
print("=== HEALTHY BASELINE ===")
print(f"  Total triads: {healthy['n_triads']}")
print(f"  Frustrated:   {healthy['n_frustrated']}")
print(f"  Ratio:        {healthy['frustration_ratio']:.4f}")
if healthy["top_frustrated"]:
    print("  Most frustrated triads (layer indices):")
    for i, j, k, c in healthy["top_frustrated"]:
        print(f"    L{i + 1}-L{j + 1}-L{k + 1}: closure error = {c:.3f} rad")

In [ ]:
# Pathological perturbations
PATHOLOGIES = {
    "epilepsy_hyper": {
        "description": "GABAergic failure → K increases 3x in L4-L5 (cortical)",
        "perturbation": lambda K: _boost_layers(K, [3, 4], 3.0),
    },
    "schizophrenia_disconnect": {
        "description": "Gamma binding deficit → K decreases 70% in L7, L10",
        "perturbation": lambda K: _reduce_layers(K, [6, 9], 0.3),
    },
    "parkinsons_dopamine": {
        "description": "DA neuron death → K decreases 80% in L2",
        "perturbation": lambda K: _reduce_layers(K, [1], 0.2),
    },
    "alzheimers_memory": {
        "description": "Tau tangles → K decreases 60% in L5, L9",
        "perturbation": lambda K: _reduce_layers(K, [4, 8], 0.4),
    },
    "autism_overconnect": {
        "description": "Local overconnectivity → K increases 2x in L3-L4, decreases 50% long-range",
        "perturbation": lambda K: _autism_pattern(K),
    },
    "meditation_enhanced": {
        "description": "Enhanced coupling → K increases 30% globally",
        "perturbation": lambda K: K * 1.3,
    },
    "psychedelic_dissolve": {
        "description": "5-HT2A agonism → omega spread increases 50%, K unchanged",
        "perturbation": lambda K: K,  # K unchanged, omega changed
        "omega_perturbation": lambda w: (
            w * (1 + 0.5 * np.random.default_rng(42).uniform(-1, 1, len(w)))
        ),
    },
}


def _boost_layers(K, layers, factor):
    K2 = K.copy()
    for layer_idx in layers:
        K2[layer_idx, :] *= factor
        K2[:, layer_idx] *= factor
    return (K2 + K2.T) / 2


def _reduce_layers(K, layers, factor):
    K2 = K.copy()
    for layer_idx in layers:
        K2[layer_idx, :] *= factor
        K2[:, layer_idx] *= factor
    return (K2 + K2.T) / 2


def _autism_pattern(K):
    K2 = K.copy()
    # Boost local (adjacent layers)
    for i in range(K.shape[0]):
        for j in range(K.shape[0]):
            if abs(i - j) <= 1 and i != j:
                K2[i, j] *= 2.0
            elif abs(i - j) >= 4:
                K2[i, j] *= 0.5
    return (K2 + K2.T) / 2


print(f"\n{'Condition':<25} {'Frustrated':>10} {'Total':>8} {'Ratio':>8} {'vs Healthy':>12}")
print("-" * 68)

pathology_results = []
for name, params in PATHOLOGIES.items():
    K_pert = params["perturbation"](K16)
    omega_pert = params.get("omega_perturbation", lambda w: w)(omega16)
    result = compute_frustration(K_pert, omega_pert)

    delta = result["frustration_ratio"] - healthy["frustration_ratio"]
    direction = "+" if delta > 0.01 else ("-" if delta < -0.01 else "=")

    print(
        f"{name:<25} {result['n_frustrated']:10d} {result['n_triads']:8d} {result['frustration_ratio']:8.4f} {delta:+11.4f} {direction}"
    )
    pathology_results.append(
        {
            "condition": name,
            "description": params["description"],
            "n_frustrated": result["n_frustrated"],
            "frustration_ratio": round(result["frustration_ratio"], 4),
            "delta_vs_healthy": round(delta, 4),
        }
    )

In [ ]:
# Simulate dynamics to confirm frustration → desync correlation
print("\n=== FRUSTRATION vs GLOBAL SYNC ===")


def quick_kuramoto_R(K, omega, T=100, dt=0.02, n_trials=5):
    N = len(omega)
    R_trials = []
    for seed in range(n_trials):
        rng = np.random.default_rng(seed)
        theta = rng.uniform(0, 2 * np.pi, N)
        n_steps = int(T / dt)
        for _s in range(n_steps):
            diff = theta[None, :] - theta[:, None]
            coupling = np.sum(K * np.sin(diff), axis=1) / N
            theta += dt * (omega + coupling) + np.sqrt(dt) * 0.05 * rng.normal(size=N)
        R_trials.append(float(np.abs(np.mean(np.exp(1j * theta)))))
    return np.mean(R_trials)


K_scale = 5.0  # above K_c for most conditions

print(f"{'Condition':<25} {'Frustration':>12} {'R (K={K_scale})':>12}")
print("-" * 52)

frust_vals = []
R_vals = []

# Healthy
R_h = quick_kuramoto_R(K16 * K_scale, omega16)
print(f"{'healthy':<25} {healthy['frustration_ratio']:12.4f} {R_h:12.3f}")
frust_vals.append(healthy["frustration_ratio"])
R_vals.append(R_h)

for name, params in PATHOLOGIES.items():
    K_pert = params["perturbation"](K16) * K_scale
    omega_pert = params.get("omega_perturbation", lambda w: w)(omega16)
    result = compute_frustration(K_pert / K_scale, omega_pert)  # frustration on unscaled
    R = quick_kuramoto_R(K_pert, omega_pert)
    print(f"{name:<25} {result['frustration_ratio']:12.4f} {R:12.3f}")
    frust_vals.append(result["frustration_ratio"])
    R_vals.append(R)

from scipy.stats import pearsonr

r_fr, p_fr = pearsonr(frust_vals, R_vals)
print(f"\nCorrelation (frustration vs R): r={r_fr:.4f}, p={p_fr:.4f}")
if r_fr < -0.3:
    print("More frustration → less sync. Frustration IS a desynchronisation mechanism.")
elif r_fr > 0.3:
    print("More frustration → more sync. Unexpected — may indicate compensatory dynamics.")
else:
    print("No clear relationship. Frustration alone does not determine sync level.")

In [ ]:
print("\n--- JSON RESULTS ---")
print(
    json.dumps(
        {
            "healthy_frustration": round(healthy["frustration_ratio"], 4),
            "healthy_n_triads": healthy["n_triads"],
            "pathologies": pathology_results,
            "frustration_vs_R_correlation": round(r_fr, 4),
            "frustration_vs_R_p": round(p_fr, 4),
        },
        indent=2,
    )
)